# Класифікація тяжкості ДТП (`Severity`): лінійна модель vs градієнтний бустинг

Дані — результат `classification_data_preparation.ipynb` (`data/processed/classification/`): ознаки вже очищені, закодовані й масштабовані, ціль `label` (0…3 = Severity 1…4). Класи сильно незбалансовані (~78% — Severity 2).

**Два сімейства моделей** (обидва — scikit-learn):
* **Сімейство 1 — `LogisticRegression`** (мультиноміальна, `lbfgs`): лінійна модель, швидка й інтерпретована.
* **Сімейство 2 — `HistGradientBoostingClassifier`**: градієнтний бустинг дерев на гістограмах ознак — враховує нелінійності й взаємодії, масштабується на мільйони записів, підтримує `class_weight`.

**Етапи:**
1. Завантаження та експрес-перевірка даних.
2. Підбір гіперпараметрів кожного сімейства на стратифікованій підвибірці train (`RandomizedSearchCV`, 5-fold CV, F1 macro).
3. Навчання на повному train: базові vs підібрані параметри × без зважування vs `class_weight="balanced"` — 4 варіанти на сімейство; таблиця метрик на train.
4. Valid: метрики всіх варіантів → лідер кожного сімейства за F1 macro → підбір порогів рішення для лідерів → матриці помилок і ROC-криві train vs valid.
5. Test: фінальна оцінка обох лідерів.
6. SHAP: beeswarm і bar chart важливості ознак для обох лідерів.

Бібліотека — scikit-learn: Spark на Windows без `winutils.exe` не читає parquet, а готові дані — звичайні parquet-файли.

## Етап 1. Завантаження та експрес-перевірка даних

In [1]:
import os
import json
import time
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

SEED = 42
# Частка train для машин з малою пам'яттю (повний train у float64 — ~9 ГБ); 1 — весь train
TRAIN_FRACTION = float(os.environ.get("TRAIN_FRACTION", "1"))
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "classification"
MODELS_DIR = PROJECT_ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES = meta["feature_names"]
LABELS = [0, 1, 2, 3]
CLASSES = [f"Severity {c}" for c in [1, 2, 3, 4]]


def load(split, dtype, fraction=1.0):
    files = sorted((DATA_DIR / split).glob("part-*.parquet"))
    rows = [pq.ParquetFile(f).metadata.num_rows for f in files]
    rng = np.random.default_rng(SEED)
    masks = [rng.random(r) < fraction if fraction < 1 else None for r in rows]
    n = sum(r if m is None else int(m.sum()) for r, m in zip(rows, masks))
    X, y, pos = np.empty((n, len(FEATURES)), dtype=dtype), np.empty(n, dtype=np.int64), 0
    for f, m in zip(files, masks):
        part = pd.read_parquet(f, columns=FEATURES + ["label"])
        if m is not None:
            part = part[m]
        X[pos:pos + len(part)] = part[FEATURES].to_numpy(dtype=dtype)
        y[pos:pos + len(part)] = part["label"].to_numpy()
        pos += len(part)
    return X, y


t0 = time.time()
X_train, y_train = load("train", np.float64, TRAIN_FRACTION)
X_valid, y_valid = load("valid", np.float32)
X_test, y_test = load("test", np.float32)
SPLITS = {"train": (X_train, y_train), "valid": (X_valid, y_valid), "test": (X_test, y_test)}
print(f"train {X_train.shape}{f' ({TRAIN_FRACTION:.0%} train)' if TRAIN_FRACTION < 1 else ''}, "
      f"valid {X_valid.shape}, test {X_test.shape} — {time.time() - t0:.0f} c")

train (4789382, 235), valid (683877, 235), test (1366450, 235) — 16 c


Матриці ознак зберігаються як numpy-масиви (train — float64, бо обидві моделі рахують у подвійній точності; valid / test — float32), тому для перегляду обгортаємо їх у `DataFrame` без копіювання. При 235 стовпцях `.info()` виводить коротке зведення (типи, пам'ять), а `.describe()` — по рядку на ознаку (показано перші й останні). Пропусків немає — перевіряємо явно.

In [2]:
for split, (X, y) in SPLITS.items():
    df = pd.DataFrame(X, columns=FEATURES, copy=False)
    print(f"\n{'=' * 30} {split}: {X.shape[0]:,} записів × {X.shape[1]} ознак {'=' * 30}")
    display(pd.DataFrame(X[:5], columns=FEATURES).assign(label=y[:5]))
    df.info()
    print(f"пропусків (NaN): {int(np.isnan(X).sum())}")
    # describe() копіює всі числові стовпці; порціями по 40 стовпців копія train — ~1.5 ГБ замість ~9 ГБ
    display(pd.concat([df.iloc[:, i:i + 40].describe() for i in range(0, X.shape[1], 40)], axis=1).T)


============================== train: 4,789,382 записів × 235 ознак ==============================


,Start_Lat,Start_Lng,Distance(mi),Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),TE_Street_t_sev1,TE_Street_t_sev3,TE_Street_t_sev4,TE_City_t_sev1,TE_City_t_sev3,TE_County_t_sev1,TE_County_t_sev3,...,"Pressure(in) in [29.86, 29.94)","Pressure(in) in [29.94, 30)","Pressure(in) in [30, 30.07)","Pressure(in) in [30.07, 30.17)","Visibility(mi) in [-inf, 5)","Visibility(mi) in [5, 10)","Wind_Speed(mph) in [0, 3)","Wind_Speed(mph) in [3, 4.6)","Wind_Speed(mph) in [4.6, 5.8)","Wind_Speed(mph) in [5.8, 7)","Wind_Speed(mph) in [7, 8)","Wind_Speed(mph) in [8, 9.2)","Wind_Speed(mph) in [9.2, 12)","Wind_Speed(mph) in [12, 14)",label
0,0.729517,0.692655,-0.579527,1.131125,-1.403600,-0.462043,0.413813,0.328066,-0.504331,1.703449,2.910487,-0.001752,1.561457,-0.003013,1.574570,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2
1,-0.415883,-1.337620,-0.579527,-0.051023,0.308934,0.549310,-0.016194,-1.372875,-0.449789,-0.899357,-0.577214,-0.826439,0.548855,-0.730986,0.413153,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.908264,1.247057,-0.579527,0.705126,-1.579245,0.222727,0.413813,0.139073,-0.593764,1.059981,0.043989,-0.796323,0.048300,-0.222845,-0.597201,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2
3,-0.238904,0.743274,-0.579527,0.119377,0.133290,-0.472578,0.413813,-1.372875,2.004260,-0.667780,-0.359061,1.818670,-0.970273,1.611694,-0.407055,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,-0.369616,0.512753,-0.579527,1.077875,-1.227956,-0.672742,0.413813,0.517060,0.037315,-0.213690,0.074158,-0.061643,-0.101272,-0.678700,0.600598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4789382 entries, 0 to 4789381
Columns: 235 entries, Start_Lat to Wind_Speed(mph) in [12, 14)
dtypes: float64(235)
memory usage: 8.4 GB


пропусків (NaN): 0


,count,mean,std,min,25%,50%,75%,max
Start_Lat,4789382.0,-4.550746e-11,1.000000,-2.312033,-0.555883,-0.086796,0.765963,2.532625
Start_Lng,4789382.0,-4.990181e-11,1.000000,-1.711357,-1.289767,0.400857,0.822363,1.603732
Distance(mi),4789382.0,8.162995e-09,1.000000,-0.579527,-0.579527,-0.557517,0.216287,4.634339
Temperature(F),4789382.0,8.937878e-10,1.000000,-2.969119,-0.679372,0.119377,0.758376,1.983124
Humidity(%),4789382.0,-7.505992e-10,1.000000,-2.457468,-0.744933,0.089378,0.835868,1.538446
...,...,...,...,...,...,...,...,...
"Wind_Speed(mph) in [5.8, 7)",4789382.0,1.266142e-01,0.332540,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [7, 8)",4789382.0,6.226899e-02,0.241643,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [8, 9.2)",4789382.0,1.320613e-01,0.338557,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [9.2, 12)",4789382.0,1.007454e-01,0.300991,0.000000,0.000000,0.000000,0.000000,1.000000


...


============================== test: 1,366,450 записів × 235 ознак ==============================


,Start_Lat,Start_Lng,Distance(mi),Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),TE_Street_t_sev1,TE_Street_t_sev3,TE_Street_t_sev4,TE_City_t_sev1,TE_City_t_sev3,TE_County_t_sev1,TE_County_t_sev3,...,"Pressure(in) in [29.86, 29.94)","Pressure(in) in [29.94, 30)","Pressure(in) in [30, 30.07)","Pressure(in) in [30.07, 30.17)","Visibility(mi) in [-inf, 5)","Visibility(mi) in [5, 10)","Wind_Speed(mph) in [0, 3)","Wind_Speed(mph) in [3, 4.6)","Wind_Speed(mph) in [4.6, 5.8)","Wind_Speed(mph) in [5.8, 7)","Wind_Speed(mph) in [7, 8)","Wind_Speed(mph) in [8, 9.2)","Wind_Speed(mph) in [9.2, 12)","Wind_Speed(mph) in [12, 14)",label
0,1.712449,0.094816,-0.579527,1.397375,-0.832755,-0.735951,0.413813,-0.049921,-0.564925,0.188288,-0.801075,-0.686431,0.472889,-0.744936,0.470710,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
1,-0.693759,-0.111344,-0.579527,1.770124,-1.227956,-0.388299,0.413813,0.517060,-0.624436,1.713051,-0.141752,-0.097877,0.597958,-0.198365,0.881437,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2
2,1.392946,0.551851,-0.579527,-0.306623,0.133290,-0.325089,0.413813,-0.805894,1.943567,-0.288639,3.653620,2.500228,-0.240422,0.830913,-0.848125,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,-0.565178,0.740149,-0.579527,0.438876,0.704134,0.201658,0.413813,-0.238914,1.521010,-0.988179,-0.785928,3.218929,-1.269534,2.942089,-1.162028,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
4,0.770898,0.684863,1.224067,0.758376,-0.876666,-0.588462,0.413813,1.462027,-0.633573,1.824494,0.892198,-0.033223,1.587222,-0.024708,1.608275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1366450 entries, 0 to 1366449
Columns: 235 entries, Start_Lat to Wind_Speed(mph) in [12, 14)
dtypes: float32(235)
memory usage: 1.2 GB


пропусків (NaN): 0


,count,mean,std,min,25%,50%,75%,max
Start_Lat,1366450.0,0.000555,1.000228,-2.312126,-0.555704,-0.086815,0.766327,2.532911
Start_Lng,1366450.0,-0.001226,1.000836,-1.715730,-1.290284,0.400348,0.822535,1.578327
Distance(mi),1366450.0,-0.001719,0.996515,-0.579527,-0.579527,-0.557517,0.214743,4.634339
Temperature(F),1366450.0,-0.002048,0.999238,-2.969119,-0.679372,0.119377,0.758376,1.983124
Humidity(%),1366450.0,0.002830,0.999642,-2.457468,-0.744933,0.089378,0.835868,1.538446
...,...,...,...,...,...,...,...,...
"Wind_Speed(mph) in [5.8, 7)",1366450.0,0.126786,0.332733,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [7, 8)",1366450.0,0.062094,0.241327,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [8, 9.2)",1366450.0,0.131903,0.338386,0.000000,0.000000,0.000000,0.000000,1.000000
"Wind_Speed(mph) in [9.2, 12)",1366450.0,0.100968,0.301287,0.000000,0.000000,0.000000,0.000000,1.000000


In [3]:
dist = pd.DataFrame({split: pd.Series(y).value_counts().reindex(LABELS) for split, (_, y) in SPLITS.items()})
dist.index = CLASSES
pd.concat([dist, (100 * dist / dist.sum()).round(2).add_suffix(", %")], axis=1)

,train,valid,test,"train, %","valid, %","test, %"
Severity 1,42475,6015,12064,0.89,0.88,0.88
Severity 2,3758419,536450,1073109,78.47,78.44,78.53
Severity 3,864430,123763,246235,18.05,18.10,18.02
Severity 4,124058,17649,35042,2.59,2.58,2.56


## Етап 2. Підбір гіперпараметрів

**Чому на підвибірці.** Пошук — це десятки навчань моделі (кандидати × 5 фолдів). На ~4.8 млн записів кожне навчання триває хвилини (логістична регресія — ~6 хв), тож повний пошук зайняв би багато годин, а 5 копій фолдів train не вмістилися б у пам'ять. Тому підбираємо на **стратифікованій підвибірці 100 000 записів (~2% train)**: стратифікація за класом зберігає частки класів, зокрема рідкісних (Severity 1 — ~0.9%, тобто ~900 записів — достатньо для стабільного F1 по класу).

**Пошук:** `RandomizedSearchCV`, `StratifiedKFold(5)`, метрика — **F1 macro** (кожен клас важить однаково — головна метрика при дисбалансі). Під час пошуку обидва сімейства використовують `class_weight="balanced"`: без зважування F1 macro на підвибірці визначається передусім тим, чи прогнозує модель рідкісні класи взагалі, а не якістю гіперпараметрів. Підібрані параметри далі застосовуються і до варіантів без зважування.

* **LogisticRegression:** сила регуляризації `C` (логарифмічна шкала 10⁻³…10²).
* **HistGradientBoosting:** `learning_rate`, кількість ітерацій `max_iter`, розмір дерев `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`, частка ознак на розбиття `max_features`. Рання зупинка вимкнена (`early_stopping=False`): на повному train вона відрізала б 10% даних копією матриці (+9 ГБ), тож кількість ітерацій — теж гіперпараметр.

Фолди й кандидати навчаються паралельно (`n_jobs=-1`, по одному потоку на процес): на 80 000 записів багатопотокові обчислення всередині однієї моделі дають більше накладних витрат, ніж користі (логістична регресія з 12 потоками BLAS навчається в ~10 разів довше, ніж з одним).

Застереження: оптимальні значення залежать від обсягу даних — на 4.8 млн записів регуляризація впливає менше, а дерева могли б бути глибшими, тож підібрані параметри радше консервативні.

In [4]:
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

TUNE_ROWS = 100_000
tune_idx = (np.arange(len(y_train)) if len(y_train) <= TUNE_ROWS else
            train_test_split(np.arange(len(y_train)), train_size=TUNE_ROWS, stratify=y_train, random_state=SEED)[0])
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]
print(f"Підвибірка для підбору: {len(tune_idx):,} записів ({100 * len(tune_idx) / len(y_train):.1f}% train)")
pd.DataFrame({"train, %": 100 * np.bincount(y_train, minlength=4) / len(y_train),
              "підвибірка, %": 100 * np.bincount(y_tune, minlength=4) / len(y_tune)}, index=CLASSES).round(3)

Підвибірка для підбору: 100,000 записів (2.1% train)


,"train, %","підвибірка, %"
Severity 1,0.887,0.887
Severity 2,78.474,78.474
Severity 3,18.049,18.049
Severity 4,2.590,2.590


In [ ]:
FAMILIES = {
    "LogReg": {"base": LogisticRegression(max_iter=300),
               "space": {"C": loguniform(1e-3, 1e2)}, "n_iter": 12},
    "HGB": {"base": HistGradientBoostingClassifier(early_stopping=False, random_state=SEED),
            "space": {"learning_rate": loguniform(0.03, 0.3), "max_iter": randint(100, 201),
                      "max_leaf_nodes": randint(15, 128), "min_samples_leaf": randint(20, 501),
                      "l2_regularization": loguniform(1e-3, 10), "max_features": uniform(0.5, 0.5)}, "n_iter": 12},
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
best_params, search_tables = {}, {}
for fam, spec in FAMILIES.items():
    t0 = time.time()
    search = RandomizedSearchCV(clone(spec["base"]).set_params(class_weight="balanced"), spec["space"],
                                n_iter=spec["n_iter"], scoring="f1_macro", cv=cv, refit=False, random_state=SEED, n_jobs=-1)
    search.fit(X_tune, y_tune)
    best_params[fam] = {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()}
    search_tables[fam] = (pd.DataFrame(search.cv_results_["params"])
                          .assign(**{"F1 macro (CV)": search.cv_results_["mean_test_score"],
                                     "std": search.cv_results_["std_test_score"]})
                          .sort_values("F1 macro (CV)", ascending=False))
    print(f"{fam}: {time.time() - t0:.0f} c, найкращий F1 macro (CV) {search.best_score_:.4f}, best_params_ = {best_params[fam]}")

In [ ]:
for fam, table in search_tables.items():
    print(f"{fam}: топ-5 кандидатів з {len(table)}")
    display(table.head(5).round(4))

## Етап 3. Навчання та первинне оцінювання моделей

На повному train навчаємо 4 варіанти кожного сімейства: **базові параметри** (значення scikit-learn за замовчуванням; для логістичної регресії — `max_iter=300` для збіжності, для бустингу — без ранньої зупинки) і **підібрані** `best_params_`, кожен **без зважування** і з **`class_weight="balanced"`** ($w_c = N / (K\,n_c)$: кожен клас дає однаковий внесок у функцію втрат → модель частіше прогнозує рідкісні класи).

**Метрики:**
* accuracy — для довідки: «завжди клас 2» вже дає ~78%;
* **balanced accuracy** — середній recall по класах;
* precision і recall кожного класу — особливо рідкісних класів 1 і 4;
* **F1 macro** (головна) і F1 weighted (зважений за кількістю записів класу);
* **ROC-AUC macro / weighted** (one-vs-rest) — якість ранжування ймовірностей незалежно від порогу.

In [ ]:
VARIANTS = {}
for fam, spec in FAMILIES.items():
    for tuned in (False, True):
        for weighted in (False, True):
            name = f"{fam} {'tuned' if tuned else 'base'}{' + balanced' if weighted else ''}"
            VARIANTS[name] = (fam, clone(spec["base"]).set_params(**(best_params[fam] if tuned else {}),
                                                                 class_weight="balanced" if weighted else None))

models, fit_info = {}, {}
for name, (fam, est) in VARIANTS.items():
    t0 = time.time()
    models[name] = est.fit(X_train, y_train)
    fit_info[name] = {"секунд": round(time.time() - t0), "ітерацій": int(np.max(est.n_iter_))}
    print(f"{name}: {fit_info[name]}")

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)


def metrics(y, proba):
    pred = proba.argmax(1)
    p, r, _, _ = precision_recall_fscore_support(y, pred, labels=LABELS, zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "balanced accuracy": balanced_accuracy_score(y, pred),
            **{f"precision {c}": p[i] for i, c in enumerate([1, 2, 3, 4])},
            **{f"recall {c}": r[i] for i, c in enumerate([1, 2, 3, 4])},
            "F1 macro": f1_score(y, pred, average="macro"), "F1 weighted": f1_score(y, pred, average="weighted"),
            "ROC-AUC macro": roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=LABELS),
            "ROC-AUC weighted": roc_auc_score(y, proba, multi_class="ovr", average="weighted", labels=LABELS)}


def metrics_table(X, y):
    table = pd.DataFrame({(VARIANTS[name][0], name): metrics(y, m.predict_proba(X)) for name, m in models.items()}).T
    return table.rename_axis(["сімейство", "модель"])


results_train = metrics_table(X_train, y_train)
results_train.round(4)

## Етап 4. Оцінювання на валідації та пороговий аналіз

Valid не бере участі в навчанні, тож на ньому порівнюємо варіанти, обираємо **лідера кожного сімейства за F1 macro** і підбираємо для лідерів пороги рішення.

In [ ]:
results_valid = metrics_table(X_valid, y_valid)
LEADERS = {fam: results_valid.loc[fam, "F1 macro"].idxmax() for fam in FAMILIES}
print("Лідери за F1 macro на valid:", LEADERS)
results_valid.round(4)

**Порогова оптимізація.** Правило «клас з найбільшою ймовірністю» при сильному дисбалансі рідко обирає рідкісні класи (у моделей без зважування) або обирає їх надто часто (зі зважуванням). Мультикласовий поріг задаємо зсувами: $\hat y = \arg\max_k (\log p_k + b_k)$. Зсуви $b_k$ підбираємо на valid так, щоб максимізувати **F1 macro** (покоординатний пошук по сітці, 3 проходи; зсув класу 2 фіксований, бо важливі лише різниці). Скориговані ймовірності — $\mathrm{softmax}(\log p + b)$: їхній argmax і є прогнозом, з них же будуємо ROC-криві.

Підбір порогів обрано замість калібрування (`CalibratedClassifierCV`): калібрування робить ймовірності чесними, але правило argmax на каліброваних ймовірностях знову майже не обирає рідкісні класи, тож порогу все одно потрібен зсув. Готової функції для мультикласових порогів немає: `TunedThresholdClassifierCV` працює лише для бінарної класифікації.

Пороги підібрано на valid, тож метрики «з порогами» на valid трохи оптимістичні; чесна оцінка — на test (етап 5).

In [ ]:
def shift(proba, offsets):
    """softmax(log p + b): скориговані ймовірності, argmax яких — прогноз з порогами."""
    z = np.log(np.clip(proba, 1e-12, 1)) + offsets
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z / z.sum(axis=1, keepdims=True)


def macro_f1(y, pred, k=4):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    denom = cm.sum(axis=0) + cm.sum(axis=1)
    return np.mean(np.where(denom > 0, 2 * np.diag(cm) / np.maximum(denom, 1), 0))


def tune_offsets(proba, y, ref=1, grid=np.linspace(-4, 4, 81), rounds=3):
    log_p = np.log(np.clip(proba, 1e-12, 1))
    offsets = np.zeros(proba.shape[1])
    best = macro_f1(y, log_p.argmax(1))
    for _ in range(rounds):
        for k in range(proba.shape[1]):
            if k == ref:
                continue
            for g in grid:
                trial = offsets.copy()
                trial[k] = g
                score = macro_f1(y, (log_p + trial).argmax(1))
                if score > best:
                    best, offsets = score, trial
    return offsets


OFFSETS, comparison = {}, {}
for fam, name in LEADERS.items():
    proba = models[name].predict_proba(X_valid)
    OFFSETS[fam] = tune_offsets(proba, y_valid)
    comparison[(fam, name)] = metrics(y_valid, proba)
    comparison[(fam, f"{name} + пороги")] = metrics(y_valid, shift(proba, OFFSETS[fam]))


def leader_proba(fam, X):
    return shift(models[LEADERS[fam]].predict_proba(X), OFFSETS[fam])


LEADER_NAMES = {fam: f"{name} + пороги" for fam, name in LEADERS.items()}
display(pd.DataFrame(OFFSETS, index=CLASSES).T.round(2).rename_axis("зсуви log-ймовірності"))
pd.DataFrame(comparison).T.rename_axis(["сімейство", "valid"]).round(4)

In [ ]:
def plot_confusion(ax, y, proba, title):
    pred = proba.argmax(1)
    ConfusionMatrixDisplay.from_predictions(y, pred, labels=LABELS, normalize="true", values_format=".2f",
                                            display_labels=[1, 2, 3, 4], cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{title}\nF1 macro = {f1_score(y, pred, average='macro'):.3f}")
    ax.set_xlabel("Прогноз Severity")
    ax.set_ylabel("Фактичний Severity")


def plot_roc(ax, curves, title):
    """curves: {мітка вибірки: (y, proba, стиль лінії)} — криві one-vs-rest кожного класу."""
    colors = sns.color_palette("tab10", 4)
    for split, (y, proba, ls) in curves.items():
        for k, cls in enumerate(CLASSES):
            fpr, tpr, _ = roc_curve(y == k, proba[:, k])
            ax.plot(fpr, tpr, ls, color=colors[k], lw=1.4, label=f"{cls}, {split}: AUC {roc_auc_score(y == k, proba[:, k]):.3f}")
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.set(title=title, xlabel="FPR", ylabel="TPR")
    ax.legend(loc="lower right", fontsize=8)


proba_leaders = {(fam, split): leader_proba(fam, X) for fam in FAMILIES
                 for split, X in [("train", X_train), ("valid", X_valid)]}

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
for row, fam in enumerate(FAMILIES):
    for col, (split, y) in enumerate([("train", y_train), ("valid", y_valid)]):
        plot_confusion(axes[row, col], y, proba_leaders[(fam, split)], f"{LEADER_NAMES[fam]} — {split}")
fig.suptitle("Матриці помилок лідерів (нормовано за фактичним класом: на діагоналі — recall)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, fam in zip(axes, FAMILIES):
    plot_roc(ax, {"train": (y_train, proba_leaders[(fam, "train")], "-"), "valid": (y_valid, proba_leaders[(fam, "valid")], "--")},
             f"{LEADER_NAMES[fam]}: ROC one-vs-rest, train (суцільна) vs valid (пунктир)")
plt.tight_layout()
plt.show()
del proba_leaders

## Етап 5. Фінальне оцінювання на тестовій вибірці

Test використовуємо один раз — для двох лідерів з порогами, підібраними на valid. Це чесна оцінка якості на нових даних.

In [ ]:
proba_test = {fam: leader_proba(fam, X_test) for fam in FAMILIES}
results_test = pd.DataFrame({(fam, LEADER_NAMES[fam]): metrics(y_test, p) for fam, p in proba_test.items()}).T
results_test.rename_axis(["сімейство", "лідер (test)"]).round(4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 14))
for col, fam in enumerate(FAMILIES):
    plot_confusion(axes[0, col], y_test, proba_test[fam], f"{LEADER_NAMES[fam]} — test")
    plot_roc(axes[1, col], {"test": (y_test, proba_test[fam], "-")}, f"{LEADER_NAMES[fam]}: ROC one-vs-rest, test")
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({(fam, split): table.loc[(fam, name), ["accuracy", "balanced accuracy", "F1 macro", "F1 weighted", "ROC-AUC macro"]]
              for fam in FAMILIES
              for split, table, name in [("train (без порогів)", results_train, LEADERS[fam]),
                                         ("valid (без порогів)", results_valid, LEADERS[fam]),
                                         ("valid (з порогами)", pd.DataFrame(comparison).T, LEADER_NAMES[fam]),
                                         ("test (з порогами)", results_test, LEADER_NAMES[fam])]}).T.round(4)

## Етап 6. SHAP-аналіз та інтерпретованість

**SHAP** розкладає прогноз на внески ознак: для кожного класу лог-шанс (margin) = базове значення + сума внесків ознак. Пороги — зсув лог-шансів поверх моделі, тож на внески ознак вони не впливають.
* **LogisticRegression:** `shap.LinearExplainer` (маскер `Independent` — середні з 2 000 записів train): внесок ознаки $j$ у клас $k$ — рівно $\beta_{kj}\,(x_j - \bar x_j)$.
* **HistGradientBoosting:** `shap.TreeExplainer` — точні SHAP-значення для дерев (з урахуванням взаємодій ознак).

**Вибірка:** 5 000 записів test, **стратифікованих за класом** — частки класів такі самі, як у test; рахувати SHAP на всіх 1.4 млн записів немає потреби.

Для кожного лідера:
* **Bar chart** — середній |SHAP| кожної ознаки, складений по класах (топ-20 стовпців матриці ознак);
* **Beeswarm** для кожного класу — кожна точка — запис, положення — внесок у лог-шанс класу, колір — значення ознаки;
* таблиця важливості **вихідних ознак** — внески one-hot категорій і бінів однієї ознаки додаються (`State`, `Distance(mi)` = значення + біни тощо).

In [ ]:
import shap

rng = np.random.default_rng(SEED)
background = X_train[rng.choice(len(X_train), size=2_000, replace=False)]
explain_idx, _ = train_test_split(np.arange(len(y_test)), train_size=5_000, stratify=y_test, random_state=SEED)
X_explain, y_explain = X_test[explain_idx].astype(np.float64), y_test[explain_idx]
print("Записів за класами у вибірці SHAP:", dict(zip(CLASSES, np.bincount(y_explain, minlength=4).tolist())))


def feature_group(name):
    if name.startswith("DoW_"):
        return "DayOfWeek"
    for sep in ("=", " in ["):
        if sep in name:
            return name.split(sep)[0]
    return name


GROUPS = [feature_group(f) for f in FEATURES]
shap_values = {}
for fam, name in LEADERS.items():
    t0 = time.time()
    model = models[name]
    explainer = (shap.LinearExplainer(model, shap.maskers.Independent(background, max_samples=len(background)))
                 if fam == "LogReg" else shap.TreeExplainer(model))
    sv = explainer(X_explain)
    sv.feature_names = FEATURES
    assert np.allclose(sv.base_values + sv.values.sum(axis=1), model.decision_function(X_explain), atol=1e-4), "база + внески ≠ лог-шанс"
    shap_values[fam] = sv
    print(f"{name}: SHAP за {time.time() - t0:.0f} c")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))
for ax, (fam, sv) in zip(axes, shap_values.items()):
    imp = pd.DataFrame(np.abs(sv.values).mean(axis=0), index=FEATURES, columns=CLASSES)
    imp.loc[imp.sum(axis=1).nlargest(20).index][::-1].plot.barh(stacked=True, ax=ax, colormap="viridis")
    ax.set_title(f"{LEADERS[fam]}: середній |SHAP| за класами, топ-20 ознак")
    ax.set_xlabel("середній |внесок| у лог-шанс класу")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
for k, ax in enumerate(axes.flat):
    shap.plots.beeswarm(shap_values["LogReg"][:, :, k], max_display=12, ax=ax, show=False, plot_size=None)
    ax.set_title(f"{LEADERS['LogReg']} — {CLASSES[k]}: внески в лог-шанс (колір — значення ознаки)")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
for k, ax in enumerate(axes.flat):
    shap.plots.beeswarm(shap_values["HGB"][:, :, k], max_display=12, ax=ax, show=False, plot_size=None)
    ax.set_title(f"{LEADERS['HGB']} — {CLASSES[k]}: внески в лог-шанс (колір — значення ознаки)")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def grouped_importance(sv):
    """Внески стовпців однієї вихідної ознаки додаються; далі середній |SHAP| по записах, сума по класах."""
    return sum(pd.DataFrame(sv.values[:, :, k], columns=FEATURES).T.groupby(GROUPS).sum().T.abs().mean()
               for k in range(len(CLASSES)))


grouped = pd.DataFrame({LEADERS[fam]: grouped_importance(sv) for fam, sv in shap_values.items()})
grouped.assign(**{f"ранг: {c}": grouped[c].rank(ascending=False).astype(int) for c in grouped.columns}) \
    .sort_values(grouped.columns[0], ascending=False).head(20).round(4)

## Збереження лідерів

In [ ]:
for fam, name in LEADERS.items():
    path = MODELS_DIR / f"classification_leader_{fam.lower()}.joblib"
    joblib.dump({"family": fam, "variant": name, "model": models[name], "log_offsets": OFFSETS[fam],
                 "params": models[name].get_params(), "features": FEATURES, "classes": "label 0..3 = Severity 1..4",
                 "predict": "argmax(log(model.predict_proba(X)) + log_offsets)"}, path)
    print("Збережено:", path.name)

## Висновки

_(заповнюється після запуску)_

**Нові дані: невідомий рік джерела.** `Source_Year` — найсильніша ознака, і вона прив'язана до років train. Перевірка «на майбутньому» (модель навчена **без 2023 р.**, оцінка на 42 167 test-записах `Source1_2023`):

| Сценарій | Accuracy | F1 класу 4 | % прогнозів класу 3 |
|---|---|---|---|
| рік був у навчанні (поточна модель) | 0.969 | 0.324 | 0% |
| невідомий рік → референсна категорія (усі `Source_Year` = 0) | 0.744 | 0.366 | 21% |
| **невідомий рік → останній рік того ж джерела** | **0.969** | 0.311 | 0% |

Тому правило для нових даних: **невідомий `Source_Year` замінюється на останній відомий рік того ж джерела** (`Source1_2023`, `Source2_2022`, `Source3_2022`; див. `metadata.json` → `unseen_source_year`). Воно працює, лише поки джерело не змінює схему розмітки (Source1 змінив її у 2021 р. — класи 1 і 3 зникли), тож на нових даних потрібні моніторинг розподілу прогнозів і періодичне перенавчання.